# Lab 9 — Structured Streaming

### The same DataFrame, now with data that never stops arriving

So far we have processed data **at rest**: files that already existed. In many cases data arrives continuously (clicks, orders, sensors, logs) and the value lies in reacting quickly.

Structured Streaming uses **the same DataFrame API** we already know. The difference is in how the query runs.

**Objectives**
- Understand the unbounded table model and micro-batches
- Learn about sources, sinks, output modes and triggers
- Aggregate by **event time** with windows
- Handle late data with **watermarks**
- Guarantee fault tolerance with **checkpoints**
- Use `availableNow` for incremental ingestion

---
## 1. Theory

### Batch vs streaming

| | Batch | Streaming |
|-|-------|-----------|
| Data | Closed set | Endless flow |
| Execution | Runs once and finishes | Runs continuously |
| Read | `spark.read` | `spark.readStream` |
| Write | `df.write` | `df.writeStream` |
| Typical latency | Minutes to hours | Seconds |

### The unbounded table model

Spark treats the flow as a **table that keeps growing**. Each new event is a new row. You write the query as if it ran over the whole table, and Spark executes it **incrementally**: it processes only what arrived and updates the result.

```
   time ────────────────────────────────────────────────────▶

   source     [e1 e2]        [e3 e4 e5]        [e6]
                 │               │               │
   input      ┌──────┐       ┌──────┐       ┌──────┐
   table      │e1 e2 │       │e1 .. │       │e1 .. │
              └──────┘       │   e5 │       │   e6 │
                 │           └──────┘       └──────┘
   query      groupBy        groupBy        groupBy     <- same query
                 │               │               │
   result     [A:2]          [A:3 B:2]      [A:3 B:3]   <- updated at each micro-batch
```

### Components of a streaming query

```
   SOURCE ──▶ TRANSFORMATIONS ──▶ SINK
   (Kafka,     (select, filter,    (files, Kafka,
    files,      groupBy, join,      Delta, memory,
    rate)       windows)            foreachBatch)
                    │
              CHECKPOINT: processed offsets + aggregation state
              TRIGGER   : when each micro-batch runs
```

### Output modes

| Mode | What is written in each micro-batch | Typical use |
|------|-------------------------------------|-----------|
| `append` | Only new rows that **will not change anymore** | Events without aggregation; windows closed by the watermark |
| `update` | Only result rows that **changed** | Dashboards, upserts |
| `complete` | The **whole** result, again | Small aggregations |

### Event time vs processing time

- **Processing time:** when Spark received the event.
- **Event time:** when the event happened (a field inside the data).

An order placed at 10:01 on a phone with no signal may arrive at 10:20. For business metrics, we almost always want **event time**. That requires dealing with **late data**, which is the watermark's job (section 5).

---
## 2. Setup

In [1]:
# !pip install -q pyspark==4.0.1

In [2]:
import os, json, time, shutil, random, uuid
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("Lab 9 - Structured Streaming")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")     # each shuffle partition becomes state; few for this lab
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.session.timeZone", "UTC")    # event and watermark times in the same time zone
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

BASE = "/tmp/lab9"
shutil.rmtree(BASE, ignore_errors=True)
os.makedirs(BASE)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/04 18:40:55 WARN Utils: Your hostname, ERJ1752, resolves to a loopback address: 127.0.1.1; using 10.23.20.50 instead (on interface enp0s31f6)
26/11/04 18:40:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 18:40:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/04 18:40:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark 4.0.1


---
## 3. First stream: the `rate` source

The `rate` source generates `N` rows per second with the columns `timestamp` and `value`. It is useful for testing because it does not depend on any external system.

The `memory` sink keeps the result in a temporary table you can query with SQL. Use it **only for learning and debugging**, because everything stays in the driver's memory.

In [3]:
rate = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 20)
    .load()
)
print("Is streaming?", rate.isStreaming)
rate.printSchema()

Is streaming? True
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)



In [4]:
events = (
    rate
    .withColumn("category", F.element_at(F.array(F.lit("Audio"), F.lit("Storage"), F.lit("Cables")),
                                         (F.col("value") % 3 + 1).cast("int")))
    .withColumn("amount", F.round(F.rand() * 100, 2))
)

q_rate = (
    events.writeStream
    .format("memory")
    .queryName("rate_events")          # name of the in-memory table
    .outputMode("append")
    .trigger(processingTime="1 second")
    .start()
)

q_rate.awaitTermination(5)             # lets it run ~5 seconds (the cell stays blocked)
spark.sql("SELECT category, COUNT(*) AS n, ROUND(SUM(amount), 2) AS total FROM rate_events GROUP BY category").show()

+--------+---+-------+
|category|  n|  total|
+--------+---+-------+
|  Cables| 33|1610.43|
|   Audio| 34|1842.06|
| Storage| 33|1447.97|
+--------+---+-------+



In [5]:
# The query keeps running in the background. Monitoring:
print("Active?    ", q_rate.isActive)
print("Status:    ", q_rate.status)

p = q_rate.lastProgress
print("Last batch:", p["batchId"], "| rows:", p["numInputRows"],
      "| rows/s processed:", round(p["processedRowsPerSecond"] or 0, 1))

Active?     True
Status:     {'message': 'Waiting for next trigger', 'isDataAvailable': True, 'isTriggerActive': False}
Last batch: 5 | rows: 20 | rows/s processed: 155.0


In [6]:
q_rate.stop()
print("Active?", q_rate.isActive)

Active? False


> Always stop the queries you no longer use. A forgotten query keeps consuming CPU and memory. `spark.streams.active` lists all the active ones.

---
## 4. File source: orders arriving in a folder

Now a more realistic scenario: a system drops JSON files with order events into a folder. Each new file is processed exactly once.

Two important cautions with file sources:
- **The schema is mandatory.** In streaming, Spark does not infer the schema by default, because it must stay stable for the entire life of the query.
- **Files must appear complete.** If Spark reads a file while it is still being written, it reads partial data. The standard approach is to write to another folder and then **move** it (a move is atomic on the same file system).

In [7]:
INPUT   = f"{BASE}/input"
STAGING = f"{BASE}/staging"
os.makedirs(INPUT); os.makedirs(STAGING)

REGIONS    = ["North", "Northeast", "Midwest", "Southeast", "South"]
CATEGORIES = ["Computers", "Peripherals", "Audio", "Storage", "Cables"]
rng = random.Random(7)

def emit(events, folder=INPUT):
    """Writes a list of events to STAGING and moves it to the input folder (atomic write)."""
    os.makedirs(folder, exist_ok=True)
    name = f"events_{uuid.uuid4().hex[:8]}.json"
    tmp = f"{STAGING}/{name}"
    with open(tmp, "w") as f:
        for e in events:
            f.write(json.dumps(e) + "\n")
    os.replace(tmp, f"{folder}/{name}")
    return len(events)

def make_event(event_time, region=None, category=None, amount=None):
    return {
        "event_id":   uuid.uuid4().hex,
        "event_time": event_time.strftime("%Y-%m-%d %H:%M:%S"),
        "order_id":   rng.randint(1, 10_000),
        "region":     region or rng.choice(REGIONS),
        "category":   category or rng.choice(CATEGORIES),
        "amount":     amount if amount is not None else round(rng.uniform(10, 500), 2),
    }

def random_batch(n, start):
    return [make_event(start + timedelta(seconds=rng.randint(0, 59))) for _ in range(n)]

event_schema = T.StructType([
    T.StructField("event_id",   T.StringType()),
    T.StructField("event_time", T.TimestampType()),
    T.StructField("order_id",   T.IntegerType()),
    T.StructField("region",     T.StringType()),
    T.StructField("category",   T.StringType()),
    T.StructField("amount",     T.DoubleType()),
])

orders_stream = (
    spark.readStream
    .schema(event_schema)
    .option("maxFilesPerTrigger", 1)       # at most 1 file per micro-batch (so we can watch it)
    .json(INPUT)
)

### 4.1 Running aggregation with `complete`

Total revenue by region since the start. At each micro-batch the whole result is rewritten.

`processAllAvailable()` blocks until all available files have been processed. It is great for labs and tests, because it makes the result deterministic. In production the query simply runs nonstop.

In [8]:
revenue_by_region = (
    orders_stream
    .groupBy("region")
    .agg(F.count("*").alias("orders"), F.round(F.sum("amount"), 2).alias("revenue"))
)

q_total = (
    revenue_by_region.writeStream
    .format("memory")
    .queryName("revenue_by_region")
    .outputMode("complete")
    .start()
)

t0 = datetime(2024, 7, 1, 10, 0, 0)
for i in range(3):
    n = emit(random_batch(50, t0 + timedelta(minutes=i)))
    q_total.processAllAvailable()
    total = spark.sql("SELECT SUM(orders) FROM revenue_by_region").first()[0]
    print(f"File {i + 1}: +{n} events -> running total = {total}")

spark.sql("SELECT * FROM revenue_by_region ORDER BY revenue DESC").show()
q_total.stop()

File 1: +50 events -> running total = 50
File 2: +50 events -> running total = 100
File 3: +50 events -> running total = 150
+---------+------+--------+
|   region|orders| revenue|
+---------+------+--------+
|    North|    42|11863.94|
|    South|    27| 7431.31|
|Northeast|    26| 7427.05|
|Southeast|    31| 6785.84|
|  Midwest|    24| 6473.14|
+---------+------+--------+



### 4.2 `update` with `foreachBatch`: what changes in each micro-batch

`foreachBatch` hands each micro-batch as a **regular DataFrame** to a function of yours. Inside it, the whole batch API is available: `show`, `write` to any format, `MERGE` into Delta, API calls.

With `update`, each micro-batch contains **only the regions that changed**.

> Avoid the `console` sink in notebooks: it prints to the Java process's output, which does not appear in the cell.

In [9]:
def show_batch(batch_df, batch_id):
    print(f"--- micro-batch {batch_id}: {batch_df.count()} rows changed")
    batch_df.orderBy("region").show()

q_update = (
    revenue_by_region.writeStream
    .foreachBatch(show_batch)
    .outputMode("update")
    .option("checkpointLocation", f"{BASE}/chk_update")
    .start()
)
q_update.processAllAvailable()     # processes the 3 files that already exist

# A new file with only South region events: only that region appears in the next micro-batch
emit([make_event(t0, region="South") for _ in range(10)])
q_update.processAllAvailable()
q_update.stop()

--- micro-batch 0: 5 rows changed
+---------+------+-------+
|   region|orders|revenue|
+---------+------+-------+
|  Midwest|     9|2343.74|
|    North|    13|3770.62|
|Northeast|     7|1803.76|
|    South|    11| 2945.6|
|Southeast|    10|1875.71|
+---------+------+-------+

--- micro-batch 1: 5 rows changed
+---------+------+-------+
|   region|orders|revenue|
+---------+------+-------+
|  Midwest|    16|4671.27|
|    North|    25|7779.21|
|Northeast|    19|5456.97|
|    South|    20|5399.06|
|Southeast|    20|4034.51|
+---------+------+-------+

--- micro-batch 2: 5 rows changed
+---------+------+--------+
|   region|orders| revenue|
+---------+------+--------+
|  Midwest|    24| 6473.14|
|    North|    42|11863.94|
|Northeast|    26| 7427.05|
|    South|    27| 7431.31|
|Southeast|    31| 6785.84|
+---------+------+--------+

--- micro-batch 3: 1 rows changed
+------+------+-------+
|region|orders|revenue|
+------+------+-------+
| South|    37|9469.01|
+------+------+-------+



---
## 5. Time windows and watermark

### Windows

`F.window(time_column, duration, slide)` groups events by **event time** intervals.

```
  Tumbling (5 min, no overlap)             Sliding (10 min, every 5 min)
  |10:00-10:05|10:05-10:10|10:10-10:15|    |10:00-------10:10|
                                                 |10:05-------10:15|
  each event falls in 1 window                         |10:10-------10:20|
                                           each event falls in 2 windows
```

### Watermark

A window can receive late events at any moment. If Spark kept the state of every window forever, memory would grow without limit. The **watermark** defines how long to wait:

```
  watermark = latest event_time seen − tolerated delay

  Example with a tolerated delay of 5 minutes:
  latest event_time seen = 10:11   ->  watermark = 10:06

  - window 10:00-10:05  -> end (10:05) < watermark: closed, emitted, state released
  - event at 10:07      -> window 10:05-10:10 still open: accepted
  - event at 10:02      -> its window has already closed: dropped
```

Two important details:
- In window aggregations, what decides whether a late event gets in is **whether its window is still open**, not the event's time compared to the watermark.
- The watermark is recomputed **at the end of each micro-batch** and applies from the next one.

With `outputMode("append")`, a window is written only **when the watermark passes its end**, because from then on the result no longer changes.

In [10]:
INPUT_W = f"{BASE}/input_windows"
os.makedirs(INPUT_W)

windowed = (
    spark.readStream.schema(event_schema).option("maxFilesPerTrigger", 1).json(INPUT_W)
    .withWatermark("event_time", "5 minutes")
    .groupBy(F.window("event_time", "5 minutes"))
    .agg(F.count("*").alias("orders"), F.round(F.sum("amount"), 2).alias("revenue"))
    .select(F.date_format("window.start", "HH:mm").alias("start"),
            F.date_format("window.end", "HH:mm").alias("end"), "orders", "revenue")
)

q_win = (
    windowed.writeStream
    .format("memory")
    .queryName("windows")
    .outputMode("append")
    .start()
)

def at(hh_mm):
    return datetime.strptime(f"2024-07-01 {hh_mm}", "%Y-%m-%d %H:%M")

last_batch = [-1]

def step(label, events):
    emit(events, INPUT_W)
    q_win.processAllAvailable()
    # a batch can produce more than one micro-batch (e.g. an extra one just to close windows)
    new = [p for p in q_win.recentProgress if p["batchId"] > last_batch[0]]
    last_batch[0] = q_win.lastProgress["batchId"]
    dropped = sum(p["stateOperators"][0]["numRowsDroppedByWatermark"] for p in new if p["stateOperators"])
    wm = q_win.lastProgress["eventTime"].get("watermark", "-")[11:16]
    print(f"{label:<40} watermark = {wm} | dropped = {dropped}")
    spark.sql("SELECT * FROM windows ORDER BY start").show()

In [11]:
# Batch 1: events between 10:00 and 10:04. No window can be closed yet.
step("Batch 1: 10:00-10:04", [make_event(at(f"10:0{m}")) for m in range(5)])

Batch 1: 10:00-10:04                     watermark = 09:59 | dropped = 0
+-----+---+------+-------+
|start|end|orders|revenue|
+-----+---+------+-------+
+-----+---+------+-------+



In [12]:
# Batch 2: events from 10:05 to 10:09 + a LATE event from 10:01 (still within the watermark)
step("Batch 2: 10:05-10:09 + late 10:01",
     [make_event(at(f"10:0{m}")) for m in range(5, 10)] + [make_event(at("10:01"))])

Batch 2: 10:05-10:09 + late 10:01        watermark = 10:04 | dropped = 0
+-----+---+------+-------+
|start|end|orders|revenue|
+-----+---+------+-------+
+-----+---+------+-------+



In [13]:
# Batch 3: events at 10:16 + a late one from 10:03.
# The current watermark is 10:04, but window 10:00-10:05 has not closed yet: the late event gets in.
step("Batch 3: 10:16 + late 10:03",
     [make_event(at("10:16")) for _ in range(3)] + [make_event(at("10:03"))])

Batch 3: 10:16 + late 10:03              watermark = 10:11 | dropped = 0
+-----+-----+------+-------+
|start|  end|orders|revenue|
+-----+-----+------+-------+
|10:00|10:05|     7|1285.13|
|10:05|10:10|     5|1561.33|
+-----+-----+------+-------+



In [14]:
# Batch 4: a VERY late event from 10:02 (its window is already closed) + an event from 10:12
step("Batch 4: very late 10:02 + 10:12", [make_event(at("10:02")), make_event(at("10:12"))])

Batch 4: very late 10:02 + 10:12         watermark = 10:11 | dropped = 1
+-----+-----+------+-------+
|start|  end|orders|revenue|
+-----+-----+------+-------+
|10:00|10:05|     7|1285.13|
|10:05|10:10|     5|1561.33|
+-----+-----+------+-------+



Reading the result:

| Batch | Current watermark | What happens |
|------|-------------------|----------------|
| 1 | none | Nothing is emitted: window 10:00-10:05 can still receive events |
| 2 | 09:59 | The late event from 10:01 is accepted. The watermark moves to 10:04, still before 10:05 |
| 3 | 10:04 | The late event from 10:03 is accepted, because its window is still open. With the 10:16 event, the watermark moves to 10:11 and windows 10:00-10:05 (7 orders) and 10:05-10:10 (5 orders) are **closed and emitted** |
| 4 | 10:11 | The 10:02 event is **dropped**: its window was already emitted. The 10:12 event goes into window 10:10-10:15, which stays open |

**The watermark trade-off:** a longer tolerated delay accepts more late events, but holds more state in memory and delays emitting results.

In [15]:
q_win.stop()

---
## 6. Writing to the data lake with a checkpoint

In production, the sink is a durable system: Parquet files, Delta or Kafka. The **checkpoint** stores:

```
  checkpoint/
  ├── offsets/    <- what each micro-batch will read   (written BEFORE processing)
  ├── commits/    <- which micro-batches finished      (written AFTER writing to the sink)
  ├── sources/    <- files already seen by the source
  └── state/      <- aggregation and window state
```

If the query goes down, restarting it with the **same checkpoint** continues where it left off: it neither loses nor duplicates data. With a sink that is also transactional (Spark files, Delta), the guarantee is **exactly-once**.

> Each query needs its own checkpoint. Deleting the checkpoint makes the query reprocess everything from the start.

In [16]:
SINK = f"{BASE}/lake/orders_bronze"
CHK  = f"{BASE}/checkpoints/orders_bronze"

def start_ingestion():
    return (
        spark.readStream.schema(event_schema).json(INPUT)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("event_date", F.to_date("event_time"))
        .writeStream
        .format("parquet")
        .option("path", SINK)
        .option("checkpointLocation", CHK)
        .partitionBy("event_date")
        .outputMode("append")
        .start()
    )

q_ing = start_ingestion()
q_ing.processAllAvailable()
print("Rows in the lake:", spark.read.parquet(SINK).count())

q_ing.stop()     # simulates a crash
print("Query stopped.")

Rows in the lake: 160
Query stopped.


In [17]:
# While the query was stopped, 2 more files arrived
emit(random_batch(40, t0)); emit(random_batch(40, t0))
files_in_input = len([f for f in os.listdir(INPUT) if f.endswith(".json")])
print("Files in the input folder:", files_in_input)

q_ing = start_ingestion()     # same checkpoint
q_ing.processAllAvailable()

n_lake   = spark.read.parquet(SINK).count()
n_unique = spark.read.parquet(SINK).select("event_id").distinct().count()
print(f"Rows in the lake: {n_lake} | distinct event_id: {n_unique}  (no duplicates)")
print("Committed micro-batches:", sorted(f for f in os.listdir(f"{CHK}/commits") if not f.startswith(".")))
q_ing.stop()

Files in the input folder: 6
Rows in the lake: 240 | distinct event_id: 240  (no duplicates)
Committed micro-batches: ['0', '1']


---
## 7. Triggers: when each micro-batch runs

| Trigger | Behavior | Use |
|---------|---------------|-----|
| (default) | New micro-batch as soon as the previous one ends | Lowest latency |
| `processingTime="1 minute"` | One micro-batch per interval | Control cost and file size |
| `availableNow=True` | Processes **everything available** and **stops** | Scheduled incremental ingestion |
| `continuous="1 second"` | Continuous processing, no micro-batch (experimental) | Millisecond latency, simple operations |

### `availableNow`: streaming as incremental ingestion

This is one of the most common uses in data lakes: it runs like a scheduled batch job (for example, every hour), but thanks to the checkpoint it **processes only the new files**. It solves the incremental Bronze problem from Lab 7 without manually tracking "which files I already read".

In [18]:
def incremental_run():
    q = (
        spark.readStream.schema(event_schema).json(INPUT)
        .writeStream.format("parquet")
        .option("path", f"{BASE}/lake/incremental")
        .option("checkpointLocation", f"{BASE}/checkpoints/incremental")
        .trigger(availableNow=True)
        .start()
    )
    q.awaitTermination()          # stops on its own when there is nothing left
    return q.lastProgress["numInputRows"] if q.lastProgress else 0

print("Run 1 — rows read:", incremental_run())
print("Run 2 — rows read:", incremental_run(), "(nothing new)")
emit(random_batch(25, t0))
print("Run 3 — rows read:", incremental_run(), "(only the new file)")
print("Total at the destination:", spark.read.parquet(f"{BASE}/lake/incremental").count())

Run 1 — rows read: 240
Run 2 — rows read: 0 (nothing new)
Run 3 — rows read: 25 (only the new file)
Total at the destination: 265


> `lastProgress` shows only the last micro-batch. If a run processes several, add up `q.recentProgress`.

---
## 8. Other streaming operations

| Operation | Support | Note |
|----------|---------|------------|
| `select`, `filter`, `withColumn` | Yes | Same as batch |
| Stream + static table join | Yes | Enrichment with dimensions |
| Stream + stream join | Yes | Requires a watermark on both sides and a time condition |
| `dropDuplicatesWithinWatermark` | Yes (3.5+) | Deduplication with bounded state |
| `orderBy` | Only after aggregation in `complete` | Sorting an infinite flow makes no sense |
| `limit`, direct `count()` | No | Use aggregations |
| `show()`, `collect()` | No | Use a sink (memory, foreachBatch) |

In [19]:
# Trying a batch action on a streaming DataFrame raises an error
try:
    orders_stream.show()
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0][:120])

AnalysisException -> Queries with streaming sources must be executed with writeStream.start();


---
## 9. Exercises

> Use `emit(events, folder)`, `make_event()`, `at()` and `processAllAvailable()` to create deterministic scenarios. For each exercise, use a new input folder and different query names.

In [20]:
# ── Exercise 1 ──────────────────────────────────────────────────────────────
# Count orders by category in 10-minute SLIDING WINDOWS that advance every 5 minutes.
# Use outputMode "complete" and a memory sink. Send events at 10:02, 10:07 and 10:12 and explain
# why each event appears in two windows.

# Your solution:
EX = f"{BASE}/ex1"; os.makedirs(EX, exist_ok=True)

sliding = (
    spark.readStream.schema(event_schema).json(EX)
    .groupBy(F.window("event_time", "10 minutes", "5 minutes"), "category")
    .count()
    .select(F.date_format("window.start", "HH:mm").alias("start"),
            F.date_format("window.end", "HH:mm").alias("end"), "category", "count")
)
q1 = sliding.writeStream.format("memory").queryName("ex1_sliding").outputMode("complete").start()

emit([make_event(at(t), category="Audio") for t in ["10:02", "10:07", "10:12"]], EX)
q1.processAllAvailable()
spark.sql("SELECT * FROM ex1_sliding ORDER BY start").show()
q1.stop()
# With a 10-min window and a 5-min step, every instant belongs to 2 windows.
# E.g. 10:07 is in [10:00, 10:10) and in [10:05, 10:15).

+-----+-----+--------+-----+
|start|  end|category|count|
+-----+-----+--------+-----+
|09:55|10:05|   Audio|    1|
|10:00|10:10|   Audio|    2|
|10:05|10:15|   Audio|    2|
|10:10|10:20|   Audio|    1|
+-----+-----+--------+-----+



In [21]:
# ── Exercise 2 ──────────────────────────────────────────────────────────────
# Stream-static join: enrich the events with the table below and compute,
# in "complete" mode, revenue by "segment".

segments = spark.createDataFrame(
    [("Computers", "Hardware"), ("Peripherals", "Hardware"), ("Storage", "Hardware"),
     ("Audio", "Accessories"), ("Cables", "Accessories")],
    ["category", "segment"],
)

# Your solution:
EX = f"{BASE}/ex2"; os.makedirs(EX, exist_ok=True)

by_segment = (
    spark.readStream.schema(event_schema).json(EX)
    .join(F.broadcast(segments), "category", "left")
    .groupBy("segment")
    .agg(F.count("*").alias("orders"), F.round(F.sum("amount"), 2).alias("revenue"))
)
q2 = by_segment.writeStream.format("memory").queryName("ex2_segments").outputMode("complete").start()

emit(random_batch(100, t0), EX)
q2.processAllAvailable()
spark.sql("SELECT * FROM ex2_segments").show()
q2.stop()

+-----------+------+--------+
|    segment|orders| revenue|
+-----------+------+--------+
|   Hardware|    56|13679.55|
|Accessories|    44|12426.19|
+-----------+------+--------+



In [22]:
# ── Exercise 3 ──────────────────────────────────────────────────────────────
# The source system sometimes resends the same event (same event_id).
# Build a stream that removes duplicates using dropDuplicatesWithinWatermark
# (10-minute watermark) and write the result to a memory sink in "append" mode.
# Prove that sending the same event 3 times makes it appear only once.

# Your solution:
EX = f"{BASE}/ex3"; os.makedirs(EX, exist_ok=True)

dedup = (
    spark.readStream.schema(event_schema).json(EX)
    .withWatermark("event_time", "10 minutes")
    .dropDuplicatesWithinWatermark(["event_id"])
)
q3 = dedup.writeStream.format("memory").queryName("ex3_dedup").outputMode("append").start()

e = make_event(at("10:00"))
emit([e], EX); emit([e, make_event(at("10:01"))], EX); emit([e], EX)     # the same event in 3 files
q3.processAllAvailable()

spark.sql("SELECT event_id, event_time, COUNT(*) AS times FROM ex3_dedup GROUP BY event_id, event_time").show(truncate=False)
q3.stop()

+--------------------------------+-------------------+-----+
|event_id                        |event_time         |times|
+--------------------------------+-------------------+-----+
|78aeecbb5af64942871595295f824cff|2024-07-01 10:01:00|1    |
|8aa512b629204e3ba63e1a87c91155d5|2024-07-01 10:00:00|1    |
+--------------------------------+-------------------+-----+



---
## Summary

| Concept | API |
|----------|-----|
| Read a stream | `spark.readStream.format(...).schema(...).load(...)` |
| Write | `df.writeStream.format(...).outputMode(...).start()` |
| Fault tolerance | `.option("checkpointLocation", path)` |
| Time window | `F.window("event_time", "5 minutes", "1 minute")` |
| Late data | `.withWatermark("event_time", "10 minutes")` |
| Free-form logic per micro-batch | `.foreachBatch(func)` |
| Incremental ingestion | `.trigger(availableNow=True)` |
| Monitor | `q.status`, `q.lastProgress`, `spark.streams.active` |
| Stop | `q.stop()` |

| Output mode | Aggregation without watermark | Aggregation with watermark | No aggregation |
|-------------|------------------------|-------------------------|---------------|
| `append` | Not supported | Closed windows | Yes |
| `update` | Yes | Yes | Yes |
| `complete` | Yes | Yes (state is not cleared) | Not supported |

In [23]:
for q in spark.streams.active:
    q.stop()
spark.stop()